<a href="https://colab.research.google.com/github/Karthik-velpula/MLOps/blob/main/231fa04509_LAB2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
data = pd.read_csv("skill_gap_dataset_realistic.csv")

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (100, 6)


,Skill_Name,Curriculum_Coverage,Industry_Demand,Job_Frequency,Expert_Importance,Gap_Status
0,Python,5,5,10,5,Aligned
1,Java,5,4,9,5,Aligned
2,C++,5,3,7,4,Aligned
3,SQL,4,5,9,5,Aligned
4,DBMS,5,4,8,5,Aligned


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Skill_Name           100 non-null    object
 1   Curriculum_Coverage  100 non-null    int64 
 2   Industry_Demand      100 non-null    int64 
 3   Job_Frequency        100 non-null    int64 
 4   Expert_Importance    100 non-null    int64 
 5   Gap_Status           100 non-null    object
dtypes: int64(4), object(2)
memory usage: 4.8+ KB


In [17]:
data.loc[0:9, "Curriculum_Coverage"] = np.nan
data.loc[15:20, "Industry_Demand"] = np.nan

In [18]:
print(data.isnull().sum())

Skill_Name              0
Curriculum_Coverage    10
Industry_Demand         6
Job_Frequency           0
Expert_Importance       0
Gap_Status              0
dtype: int64


In [19]:
features = [
    "Skill_Name",
    "Curriculum_Coverage",
    "Industry_Demand",
    "Job_Frequency",
    "Expert_Importance"
]

target = "Gap_Status"

X = data[features]
y = data[target]

print(X.head())
print(y.head())

  Skill_Name  Curriculum_Coverage  Industry_Demand  Job_Frequency  \
0     Python                  NaN              5.0             10   
1       Java                  NaN              4.0              9   
2        C++                  NaN              3.0              7   
3        SQL                  NaN              5.0              9   
4       DBMS                  NaN              4.0              8   

   Expert_Importance  
0                  5  
1                  5  
2                  4  
3                  5  
4                  5  
0    Aligned
1    Aligned
2    Aligned
3    Aligned
4    Aligned
Name: Gap_Status, dtype: object


In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

Training records: 80
Testing records: 20


In [21]:
numerical_only = [
    "Curriculum_Coverage",
    "Industry_Demand",
    "Job_Frequency",
    "Expert_Importance"
]

baseline_train = X_train[numerical_only].copy()
baseline_test = X_test[numerical_only].copy()

train_valid_rows = baseline_train.dropna().index
test_valid_rows = baseline_test.dropna().index

baseline_train = baseline_train.loc[train_valid_rows]
baseline_test = baseline_test.loc[test_valid_rows]

y_train_baseline = y_train.loc[train_valid_rows]
y_test_baseline = y_test.loc[test_valid_rows]

baseline_model = LogisticRegression(max_iter=1000)

baseline_model.fit(baseline_train, y_train_baseline)

baseline_predictions = baseline_model.predict(baseline_test)

baseline_accuracy = accuracy_score(
    y_test_baseline,
    baseline_predictions
)

print("Baseline accuracy:", baseline_accuracy)
print("Training records used:", len(baseline_train))
print("Testing records used:", len(baseline_test))

Baseline accuracy: 0.7647058823529411
Training records used: 67
Testing records used: 17


In [22]:
numerical_features = [
    "Curriculum_Coverage",
    "Industry_Demand",
    "Job_Frequency",
    "Expert_Importance"
]

categorical_features = [
    "Skill_Name"
]

In [23]:
numerical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [24]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [26]:
complete_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(max_iter=1000)
        )
    ]
)

In [27]:
complete_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numerical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Curriculum_Coverage',
                                                   'Industry_Demand',
                                                   'Job_Frequency',
                                                   'Expert_Importance']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Skill_Name'])])),
                ('model', LogisticRegression(max_iter=1000))])

In [28]:
processed_predictions = complete_pipeline.predict(X_test)

processed_accuracy = accuracy_score(
    y_test,
    processed_predictions
)

print("Accuracy after preprocessing:", processed_accuracy)

Accuracy after preprocessing: 0.8


In [29]:
print(
    classification_report(
        y_test,
        processed_predictions
    )
)

              precision    recall  f1-score   support

     Aligned       0.88      0.70      0.78        10
         Gap       0.75      0.90      0.82        10

    accuracy                           0.80        20
   macro avg       0.81      0.80      0.80        20
weighted avg       0.81      0.80      0.80        20



In [30]:
comparison = pd.DataFrame(
    {
        "Approach": [
            "Before proper preprocessing",
            "After preprocessing pipeline"
        ],
        "Accuracy": [
            baseline_accuracy,
            processed_accuracy
        ],
        "Training records used": [
            len(baseline_train),
            len(X_train)
        ],
        "Testing records used": [
            len(baseline_test),
            len(X_test)
        ]
    }
)

print(comparison)

                       Approach  Accuracy  Training records used  \
0   Before proper preprocessing  0.764706                     67   
1  After preprocessing pipeline  0.800000                     80   

   Testing records used  
0                    17  
1                    20  


In [31]:
X_train_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_train)

X_test_processed = complete_pipeline.named_steps[
    "preprocessing"
].transform(X_test)

In [32]:
feature_names = complete_pipeline.named_steps[
    "preprocessing"
].get_feature_names_out()

print(feature_names)

['numerical__Curriculum_Coverage' 'numerical__Industry_Demand'
 'numerical__Job_Frequency' 'numerical__Expert_Importance'
 'categorical__Skill_Name_AWS' 'categorical__Skill_Name_Azure'
 'categorical__Skill_Name_C++' 'categorical__Skill_Name_Cloud Computing'
 'categorical__Skill_Name_Communication'
 'categorical__Skill_Name_Computer Networks'
 'categorical__Skill_Name_Cyber Security' 'categorical__Skill_Name_DBMS'
 'categorical__Skill_Name_Data Analytics'
 'categorical__Skill_Name_Deep Learning' 'categorical__Skill_Name_DevOps'
 'categorical__Skill_Name_Docker' 'categorical__Skill_Name_Generative AI'
 'categorical__Skill_Name_Git' 'categorical__Skill_Name_Java'
 'categorical__Skill_Name_Kubernetes' 'categorical__Skill_Name_Linux'
 'categorical__Skill_Name_Machine Learning'
 'categorical__Skill_Name_NodeJS'
 'categorical__Skill_Name_Operating Systems'
 'categorical__Skill_Name_Problem Solving'
 'categorical__Skill_Name_Python' 'categorical__Skill_Name_React'
 'categorical__Skill_Name_SQL

In [33]:
processed_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

processed_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [34]:
processed_train_df["Gap_Status"] = y_train.values
processed_test_df["Gap_Status"] = y_test.values

In [35]:
processed_train_df.head()

,numerical__Curriculum_Coverage,numerical__Industry_Demand,numerical__Job_Frequency,numerical__Expert_Importance,categorical__Skill_Name_AWS,categorical__Skill_Name_Azure,categorical__Skill_Name_C++,categorical__Skill_Name_Cloud Computing,categorical__Skill_Name_Communication,categorical__Skill_Name_Computer Networks,...,categorical__Skill_Name_Linux,categorical__Skill_Name_Machine Learning,categorical__Skill_Name_NodeJS,categorical__Skill_Name_Operating Systems,categorical__Skill_Name_Problem Solving,categorical__Skill_Name_Python,categorical__Skill_Name_React,categorical__Skill_Name_SQL,categorical__Skill_Name_Teamwork,Gap_Status
56,0.682288,-0.893765,-0.919494,-1.732051,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Aligned
65,-0.833908,0.695151,0.023577,0.577350,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Gap
49,-0.075810,0.695151,-0.919494,0.577350,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Gap
10,-0.833908,0.695151,0.966648,0.577350,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Gap
24,-0.075810,0.695151,0.023577,0.577350,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Gap


In [36]:
processed_train_df.to_csv(
    "skill_gap_processed_train.csv",
    index=False
)

processed_test_df.to_csv(
    "skill_gap_processed_test.csv",
    index=False
)

In [37]:
from google.colab import files

files.download("skill_gap_processed_train.csv")
files.download("skill_gap_processed_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
joblib.dump(
    complete_pipeline,
    "skill_gap_preprocessing_model_pipeline.pkl"
)

['skill_gap_preprocessing_model_pipeline.pkl']

In [39]:
files.download(
    "skill_gap_preprocessing_model_pipeline.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
loaded_pipeline = joblib.load(
    "skill_gap_preprocessing_model_pipeline.pkl"
)

In [41]:
new_skill = pd.DataFrame(
    {
        "Skill_Name": ["Python"],
        "Curriculum_Coverage": [70],
        "Industry_Demand": [90],
        "Job_Frequency": [85],
        "Expert_Importance": [95]
    }
)

In [42]:
prediction = loaded_pipeline.predict(new_skill)

if prediction[0] == 1:
    print("Predicted result: Skill Gap Exists")
else:
    print("Predicted result: No Skill Gap")

Predicted result: No Skill Gap
